# Neural Network for Rating Search Query Relevance

**Author** : Huy Le

## Project goal
 

This project builds a machine-learning model that assigns a quality rating
to a user search query.

 
The rating is a decimal number between 0 (vague, unclear) and 1 (clear, specific).

The project uses two CSV files: 
- `train.csv`: contains queries and known ratings.
- `test.csv`: contains queries whose ratings must be predicted.
 

The notebook compares two neural-network architectures:
1. An embedding model with average pooling.
2. An embedding model with a bidirectional GRU (gated recurrent unit).
 

The models are evaluated using Mean Absolute Error, abbreviated as MAE. A smaller MAE represents more accurate predictions.
 
## Important data-handling rule
 

The notebook does not add columns to or remove columns from the original training and testing DataFrames.

 
All cleaned text, query lengths, model inputs, and predictions are stored in separate Python variables.

## 1. Imports, reproducibility, and configuration

Seeds improve repeatability. Change `DATA_DIR` only if the three CSV files are not in the notebook folder.

In [ ]:
"""
This module imports all Python libraries required by the project and defines
the main configuration values used throughout the notebook.

Why this module must run first
------------------------------
Later modules refer to names such as pd, np, tf, TRAIN_PATH, and SEED.
Those names do not exist until this module has been executed.

-------------------------
A Python library is a collection of tools written by other developers.
Instead of writing every operation from scratch, we import libraries that
already know how to:

* read CSV files,
* organize information into tables,
* create graphs,
* split information into training and validation groups,
* build neural networks,
* measure prediction error, and
* save models and predictions.

Source-data protection
----------------------
Nothing in this module changes train.csv or test.csv. It only defines where
those files are expected to be found.
"""

# Standard Python libraries
import os
import re
import random
import warnings
from pathlib import Path

# Data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Traditional machine-learning tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# Neural-network tools
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# Hide noncritical warning messages so the notebook is easier to read.
warnings.filterwarnings("ignore")


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

# A seed makes random operations more repeatable.
#
# The exact numerical results can still vary slightly across computers,
# operating systems, TensorFlow versions, or hardware types.
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ---------------------------------------------------------
# File locations
# ---------------------------------------------------------

# The project expects train.csv and test.csv inside the data folder.
DATA_DIR = Path("data")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# Generated outputs are placed in a separate outputs folder.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
MODEL_PATH = OUTPUT_DIR / "best_query_rating_model.keras"
RESULTS_PATH = OUTPUT_DIR / "model_experiment_results.csv"


# ---------------------------------------------------------
# Model configuration
# ---------------------------------------------------------

# Twenty percent of the labeled information is reserved for validation.
VALIDATION_SIZE = 0.20

# Number of observations processed before a model-weight update.
BATCH_SIZE = 64

# Maximum number of passes through the training information.
# Early stopping may end training before this number is reached.
MAX_EPOCHS = 25

# Maximum number of vocabulary tokens retained by the neural vectorizer.
MAX_TOKENS = 20_000

# Number of numerical values used to represent each learned token.
EMBEDDING_DIM = 64


print("Configuration completed successfully.")
print("TensorFlow version:", tf.__version__)
print("Pandas version:", pd.__version__)
print("Training path:", TRAIN_PATH)
print("Testing path:", TEST_PATH)
print("Output folder:", OUTPUT_DIR.resolve())

## 2. Load and validate the files

The checks below fail early with a clear message if a required file or field is absent. The original order of `test.csv` is preserved for submission alignment.

In [ ]:
"""

Purpose
-------
This module reads train.csv and test.csv into memory and verifies that their
column structures match the project requirements.

Variables created
-----------------
data:
    A Pandas DataFrame created from train.csv. It should contain the columns
    query and rating.

test_data:
    A Pandas DataFrame created from test.csv. It should contain only query.

What is a DataFrame?
--------------------
A DataFrame is a table in Python. It resembles a spreadsheet with rows and
columns.

Important source-file behavior
------------------------------
pd.read_csv reads a file into computer memory. Reading the file does not
change the CSV stored on disk.

This module also avoids adding, deleting, or renaming columns.
"""

# Confirm that both required files exist.
missing_files = [
    str(path)
    for path in [TRAIN_PATH, TEST_PATH]
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required file or files could not be found: "
        + ", ".join(missing_files)
        + ". Confirm that the data folder contains train.csv and test.csv."
    )


# Read the original CSV files into Pandas DataFrames.
data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)


# Define the exact schemas required by the project.
expected_train_columns = ["query", "rating"]
expected_test_columns = ["query"]


# Require exact column names and exact column order.
if list(data.columns) != expected_train_columns:
    raise ValueError(
        "train.csv must contain exactly these columns in this order: "
        f"{expected_train_columns}. Found: {data.columns.tolist()}"
    )

if list(test_data.columns) != expected_test_columns:
    raise ValueError(
        "test.csv must contain exactly this column: "
        f"{expected_test_columns}. Found: {test_data.columns.tolist()}"
    )


# Save the original structure for a verification check near the end.
original_train_columns = tuple(data.columns)
original_test_columns = tuple(test_data.columns)

original_train_shape = data.shape
original_test_shape = test_data.shape


print("The datasets were loaded successfully.")
print("Training shape:", data.shape)
print("Testing shape:", test_data.shape)

print("\nFirst five training rows:")
display(data.head())

print("\nFirst five testing rows:")
display(test_data.head())

## 3. Create modeling arrays without changing the DataFrames

This module extracts query text and ratings into separate NumPy arrays.

In [ ]:
"""
Why separate arrays are used
----------------------------
The original DataFrames should remain unchanged. Therefore, this notebook
does not add columns such as query_clean, word_count, or prediction to data
or test_data.

Instead, it creates separate variables:

all_train_queries:
    Cleaned versions of all queries from train.csv.

all_train_ratings:
    Numerical versions of all ratings from train.csv.

model_queries:
    Queries whose rows contain usable ratings.

model_ratings:
    Valid numerical ratings restricted to the interval from 0 to 1.

test_queries:
    Cleaned queries from test.csv.

Meaning of query cleaning
-------------------------
The cleaning function:

* replaces missing query values with an empty string,
* converts each query to text,
* replaces repeated whitespace with one space, and
* removes whitespace from the beginning and end.

It does not aggressively delete words, numbers, or punctuation.
"""


def clean_query(value):
    """
    Convert one query into a consistent text value.

    Parameters
    ----------
    value:
        One value from the query column. It may be text, a number, a missing
        value, or another value that can be converted to text.

    Returns
    -------
    str
        A cleaned string.

    Examples
    --------
    "  learn   Python  " becomes "learn Python".

    A missing value becomes an empty string.

    Why re.sub is used
    ------------------
    re is Python's regular-expression library. The pattern r"\\s+" means
    "one or more whitespace characters." Those characters are replaced by
    one ordinary space.
    """

    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


# Clean the query columns without assigning anything back to the DataFrames.
all_train_queries = (
    data["query"]
    .map(clean_query)
    .to_numpy()
)

test_queries = (
    test_data["query"]
    .map(clean_query)
    .to_numpy()
)


# Convert ratings to numbers in a separate object.
# Unreadable values become NaN, meaning "not a number."
all_train_ratings = pd.to_numeric(
    data["rating"],
    errors="coerce"
).to_numpy(dtype="float32")


# Identify which ratings are usable.
valid_rating_mask = ~np.isnan(all_train_ratings)

invalid_rating_count = int(
    (~valid_rating_mask).sum()
)


# Filter only the temporary modeling arrays.
# No row is removed from the original data DataFrame.
model_queries = all_train_queries[valid_rating_mask]
model_ratings = all_train_ratings[valid_rating_mask]


# Count ratings outside the required interval before clipping them.
out_of_range_count = int(
    (
        (model_ratings < 0)
        | (model_ratings > 1)
    ).sum()
)

# Keep the temporary model ratings between 0 and 1.
model_ratings = np.clip(
    model_ratings,
    0.0,
    1.0
).astype("float32")


print("Original training rows:", len(data))
print("Usable modeling rows:", len(model_queries))
print("Invalid ratings excluded from modeling:", invalid_rating_count)
print("Out-of-range ratings clipped:", out_of_range_count)
print("Test queries:", len(test_queries))

## 4. Explore ratings and query lengths

This module summarizes and visualizes the synthetic training information.

In [ ]:
"""
Why exploration matters
-----------------------
A model should not be trained without first checking the information it will
learn from. Exploration can reveal:

* whether ratings cover the full 0-to-1 range,
* whether most queries are unusually short or long,
* whether query length appears related to rating, and
* whether a few very long queries could cause excessive sequence padding.

No source columns are created
-----------------------------
Word and character counts are stored in independent NumPy arrays. They are
not added to data.
"""

# Count whitespace-separated words in every usable training query.
word_counts = np.array(
    [
        len(query.split())
        for query in model_queries
    ]
)

# Count characters in every usable training query.
character_counts = np.array(
    [
        len(query)
        for query in model_queries
    ]
)


print("Rating summary:")
display(
    pd.Series(
        model_ratings,
        name="rating"
    ).describe()
)

print("\nWord-count summary:")
display(
    pd.Series(
        word_counts,
        name="word_count"
    ).describe()
)

print("\nCharacter-count summary:")
display(
    pd.Series(
        character_counts,
        name="character_count"
    ).describe()
)


# Create three visual summaries.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4)
)

axes[0].hist(
    model_ratings,
    bins=20,
    edgecolor="black"
)
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Number of queries")

axes[1].hist(
    word_counts,
    bins=30,
    edgecolor="black"
)
axes[1].set_title("Query length distribution")
axes[1].set_xlabel("Number of words")
axes[1].set_ylabel("Number of queries")

axes[2].scatter(
    word_counts,
    model_ratings,
    alpha=0.25,
    s=12
)
axes[2].set_title("Rating versus query length")
axes[2].set_xlabel("Number of words")
axes[2].set_ylabel("Rating")

plt.tight_layout()
plt.show()


# Determine the length that covers approximately 95 percent of queries.
length_95 = int(
    np.ceil(
        np.percentile(
            word_counts,
            95
        )
    )
)

# Use practical lower and upper limits.
SEQUENCE_LENGTH = int(
    np.clip(
        length_95,
        10,
        100
    )
)

print("95th-percentile query length:", length_95)
print("Neural-network sequence length:", SEQUENCE_LENGTH)

## 5. Split the labeled information

This module divides the labeled information into two groups:

- Training group:
    Used by the model to learn relationships between query text and ratings.

- Validation group:
    Hidden from the learning process and used to estimate how well the model
    handles observations it did not train on.

In [ ]:
"""
Why not train on everything immediately?
----------------------------------------
A model can appear accurate if it is evaluated on the same information that
it memorized. A validation group provides a more meaningful test.

The test.csv file is not used here
----------------------------------
The test queries do not have known ratings. They are saved for final
prediction after model selection.
"""

X_train, X_valid, y_train, y_valid = train_test_split(
    model_queries,
    model_ratings,
    test_size=VALIDATION_SIZE,
    random_state=SEED
)


print("Training observations:", len(X_train))
print("Validation observations:", len(X_valid))
print("Final test observations:", len(test_queries))

print(
    "Average training rating:",
    round(float(np.mean(y_train)), 4)
)

print(
    "Average validation rating:",
    round(float(np.mean(y_valid)), 4)
)

## 6. Training-only text vectorization and model definitions

The vocabulary is adapted only on the training split to prevent preprocessing leakage. `AveragePoolingNN` is compact, while `BidirectionalGRUNN` models token order. Both use sigmoid output, Adam, MAE loss, dropout, and L2 regularization.

In [ ]:
# Learn integer token IDs from training text only.
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation", split="whitespace",
    name="text_vectorizer"
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))
VOCAB_SIZE = len(vectorizer.get_vocabulary())
print("Vocabulary size:", VOCAB_SIZE)

def compile_model(model):
    # MAE directly matches the evaluation requirement.
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.MeanAbsoluteError(),
                  metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
    return model

def build_average_model():
    text = keras.Input(shape=(), dtype=tf.string, name="query")
    x = vectorizer(text)
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    output = layers.Dense(1, activation="sigmoid", name="rating")(x)
    return compile_model(keras.Model(text, output, name="average_pooling_nn"))

def build_bigru_model():
    text = keras.Input(shape=(), dtype=tf.string, name="query")
    x = vectorizer(text)
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(x)
    x = layers.Bidirectional(layers.GRU(48, dropout=0.20))(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.35)(x)
    output = layers.Dense(1, activation="sigmoid", name="rating")(x)
    return compile_model(keras.Model(text, output, name="bidirectional_gru_nn"))

model_builders = {"AveragePoolingNN": build_average_model,
                  "BidirectionalGRUNN": build_bigru_model}
build_average_model().summary()

## 7. Train, control overfitting, and select the best neural network

Early stopping restores the weights from the lowest validation loss. Learning-rate reduction allows smaller updates after a plateau. Learning curves make overfitting visible, and scikit-learn MAE provides the final holdout score.

In [ ]:
histories, trained_models, rows = {}, {}, []

for name, builder in model_builders.items():
    print("\n" + "=" * 75 + f"\nTraining {name}")
    tf.keras.utils.set_random_seed(SEED)
    model = builder()
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
            min_delta=1e-4, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
            patience=2, min_lr=1e-6, verbose=1)
    ]
    history = model.fit(X_train, y_train,
        validation_data=(X_valid, y_valid), epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)
    pred = np.clip(model.predict(X_valid, batch_size=BATCH_SIZE,
                                 verbose=0).reshape(-1), 0, 1)
    mae = mean_absolute_error(y_valid, pred)
    histories[name] = history.history
    trained_models[name] = model
    rows.append({"model": name, "validation_mae": mae,
                 "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
                 "parameters": model.count_params()})

results_df = pd.DataFrame(rows).sort_values("validation_mae").reset_index(drop=True)
display(results_df)

# Plot train and validation MAE. A widening gap suggests overfitting.
fig, axes = plt.subplots(1, len(histories), figsize=(7 * len(histories), 4), squeeze=False)
for ax, (name, history) in zip(axes.ravel(), histories.items()):
    epochs = np.arange(1, len(history["loss"]) + 1)
    ax.plot(epochs, history["loss"], marker="o", label="Train MAE")
    ax.plot(epochs, history["val_loss"], marker="o", label="Validation MAE")
    ax.set(title=name, xlabel="Epoch", ylabel="MAE"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

best_name = results_df.loc[0, "model"]
best_model = trained_models[best_name]
best_nn_mae = float(results_df.loc[0, "validation_mae"])
comparison = pd.DataFrame({"model":["Mean baseline", "TF-IDF + Ridge", best_name],
                           "validation_mae":[mean_mae, ridge_mae, best_nn_mae]})
print("Selected neural network:", best_name)
display(comparison.sort_values("validation_mae"))

## 8. Validation diagnostics

The diagonal represents perfect predictions. Residuals should be centered near zero and should not show a strong pattern against predicted ratings.

In [ ]:
valid_pred = np.clip(best_model.predict(X_valid, batch_size=BATCH_SIZE,
                                                verbose=0).reshape(-1), 0, 1)
residuals = y_valid - valid_pred
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_valid, valid_pred, alpha=.4); axes[0].plot([0,1],[0,1], "k--")
axes[0].set(title="Actual versus predicted", xlabel="Actual", ylabel="Predicted")
axes[1].hist(residuals, bins=25, edgecolor="black"); axes[1].axvline(0, color="k", ls="--")
axes[1].set(title="Residual distribution", xlabel="Actual - predicted", ylabel="Count")
axes[2].scatter(valid_pred, residuals, alpha=.4); axes[2].axhline(0, color="k", ls="--")
axes[2].set(title="Residuals versus predictions", xlabel="Predicted", ylabel="Residual")
plt.tight_layout(); plt.show()
print(f"Validation MAE: {mean_absolute_error(y_valid, valid_pred):.6f}")
print(f"Mean residual: {residuals.mean():.6f}")

## 9. Refit on all labeled data, predict, and create the deliverables

After architecture selection, preprocessing and the selected model are rebuilt on all unique labeled queries. The best holdout epoch count is reused instead of the maximum. Predictions are clipped to `[0, 1]`, the exact test query order is retained, and the template schema is checked.

In [ ]:
all_queries = model_df.query_clean.astype(str).to_numpy()
all_ratings = model_df.rating.astype("float32").to_numpy()

# Relearn vocabulary from all labeled text for the final production model.
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation", split="whitespace",
    name="text_vectorizer")
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(all_queries).batch(256))
VOCAB_SIZE = len(vectorizer.get_vocabulary())
selected_epochs = int(results_df.loc[0, "best_epoch"])

# Builder functions use the newly fitted vectorizer and vocabulary size.
tf.keras.utils.set_random_seed(SEED)
final_model = model_builders[best_name]()
final_model.fit(all_queries, all_ratings, epochs=selected_epochs,
                batch_size=BATCH_SIZE, verbose=1)

test_pred = np.clip(final_model.predict(X_test, batch_size=BATCH_SIZE,
                                        verbose=0).reshape(-1), 0, 1)
submission_output = pd.DataFrame({"query": test_df["query"],
                                  "rating": test_pred.astype(float)})

if TEMPLATE_PATH.exists():
    template = pd.read_csv(TEMPLATE_PATH)
    if list(template.columns) != ["query", "rating"]:
        raise ValueError("submission.csv must have exactly query and rating columns")
    if len(template) != len(submission_output):
        raise ValueError("submission.csv and test.csv row counts differ")

# Final safety checks before saving.
assert list(submission_output.columns) == ["query", "rating"]
assert len(submission_output) == len(test_df)
assert submission_output.rating.notna().all()
assert submission_output.rating.between(0, 1).all()

submission_output.to_csv(OUTPUT_PATH, index=False)
final_model.save(DATA_DIR / "best_query_rating_model.keras")
results_df.to_csv(DATA_DIR / "model_experiment_results.csv", index=False)
print("Saved submission:", OUTPUT_PATH.resolve())
display(submission_output.head())

## Interpretation

MAE is the average absolute difference between true and predicted ratings. It is nonnegative, and **lower is better**, with zero representing perfect predictions.

Useful next experiments include character-level neural inputs, repeated cross-validation, pretrained language representations, and tuning vocabulary size, embedding width, GRU width, dropout, and learning rate. Avoid repeatedly tuning against one holdout set because model-selection decisions can eventually overfit it.